# CISC 867 Final Project: SimCLR & SupCon (Group 20)


This notebook consolidates the entire codebase into a single, reproducible environment. It maps directly to the experiments presented in the Final Report, including the midterm ablations, the champion 1001-epoch ResNet-50 run (Exp 41), Supervised Contrastive Learning (SupCon), and CLIP Zero-Shot evaluation.



## 1. Environment & Imports


In [1]:
import os
import sys
import time
import math
import json
import csv
import random
import argparse
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
from torch.utils.data import DataLoader, Dataset, Subset, TensorDataset

import torchvision
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torchvision.datasets import CIFAR10

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


Using device: cuda
GPU: NVIDIA RTX 5000 Ada Generation
VRAM: 34.4 GB


## 2. Augmentations Registry
Contains the 42 experiment pipelines (including Exp 41 Champion).


In [2]:

# IMPORT PART
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torch.nn.functional as F
import torch
import random

# FIXED — hardcode directly (no YAML dependency):
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)
s          = 1.0   # color distortion strength

# SIMCLR VIEW GENERATOR CLASS 
class SimCLRViewGenerator(object):
    """
    A custom wrapper to generate multiple augmented views of a single image,
    following the SimCLR framework for contrastive learning.
    """
    def __init__(self, base_transform, n_views=2):
        # The core stochastic augmentation pipeline to be applied to each input.
        self.base_transform = base_transform
        # Number of augmented views (typically 2 for creating positive pairs in SimCLR).
        self.n_views = n_views

    def __call__(self, x):
        """
        Executes the transformation pipeline multiple times to produce different 
        stochastic realizations of the same input image 'x'.
        """
        # Returns a list of 'n_views' variations, facilitating the calculation of contrastive loss.
        return [self.base_transform(x) for i in range(self.n_views)]


color_jitter = T.ColorJitter(0.8*s, 0.8*s, 0.8*s, 0.2*s)
# Applying jitter with 0.8 probability as per Chen et al. (2020)
rnd_color_jitter = T.RandomApply([color_jitter], p=0.8)


# EXPERIMENT 1 — Random Resized Crop (Crop + Resize)
transform_exp1 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)), 
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# EXP 2 — Crop + Flip
transform_exp2 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# EXP 3 — Crop + Color
transform_exp3 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    rnd_color_jitter,
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# EXP 4 — Crop + Grayscale
transform_exp4 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomGrayscale(p=0.2),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# EXP 5 — Crop + Flip + Color
transform_exp5 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    rnd_color_jitter,
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# EXP 6 — Crop + Flip + Grayscale
transform_exp6 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomGrayscale(p=0.2),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# EXP 7 — Crop + Color + Grayscale
transform_exp7 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    rnd_color_jitter,
    T.RandomGrayscale(p=0.2),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# EXP 8 — Crop + Flip + Color + Grayscale
transform_exp8 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    rnd_color_jitter,
    T.RandomGrayscale(p=0.2),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# ============================================================================
# NATALIE'S 28 EXPERIMENTS (from final augmentation.py)
# ============================================================================

class SobelTransform:
    def __call__(self, img):
        if not torch.is_tensor(img):
            img = TF.to_tensor(img)
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]).float().view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]).float().view(1, 1, 3, 3)
        gray = img.mean(dim=0, keepdim=True).unsqueeze(0)
        gx = F.conv2d(gray, sobel_x, padding=1)
        gy = F.conv2d(gray, sobel_y, padding=1)
        edge = torch.sqrt(gx**2 + gy**2).squeeze(0)
        return edge.repeat(3, 1, 1)

class GaussianNoise:
    def __init__(self, std=0.1): 
        self.std = std
    def __call__(self, img):
        if not torch.is_tensor(img): 
            img = TF.to_tensor(img)
        noise = torch.randn_like(img) * self.std
        return (img + noise).clamp(0, 1)

class DiscreteRotation:
    def __call__(self, img):
        angles = [0, 90, 180, 270]
        return TF.rotate(img, random.choice(angles))

nat_transform_exp1 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp2 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.7, scale=(0.02, 0.2), ratio=(0.3, 3.3))])
nat_transform_exp3 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), SobelTransform(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp4 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), GaussianNoise(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])

nat_transform_exp5 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp6 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.5)])
nat_transform_exp7 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), SobelTransform(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp8 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), GaussianNoise(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp9 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), rnd_color_jitter, T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp10 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), rnd_color_jitter, T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.5)])
nat_transform_exp11 = T.Compose([T.RandomResizedCrop(32), rnd_color_jitter, T.ToTensor(), SobelTransform(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp12 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), rnd_color_jitter, GaussianNoise(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp13 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomGrayscale(p=0.2), T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp14 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomGrayscale(p=0.2), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.5)])
nat_transform_exp15 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomGrayscale(p=0.2), SobelTransform(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp16 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomGrayscale(p=0.2), GaussianNoise(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])

nat_transform_exp17 = T.Compose([T.RandomResizedCrop(32), T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.ToTensor(), SobelTransform(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp18 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), T.ToTensor(), T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.5)])
nat_transform_exp19 = T.Compose([T.RandomResizedCrop(32), T.ToTensor(), SobelTransform(), GaussianNoise(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp20 = T.Compose([T.RandomResizedCrop(32), T.GaussianBlur(3), T.ToTensor(), GaussianNoise(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp21 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), rnd_color_jitter, T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.5)])
nat_transform_exp22 = T.Compose([T.RandomResizedCrop(32), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.9, scale=(0.05, 0.4), ratio=(0.3, 3.3))])

nat_transform_exp23 = T.Compose([T.RandomResizedCrop(32, scale=(0.08, 1.0)), T.RandomHorizontalFlip(), rnd_color_jitter, T.RandomGrayscale(p=0.2), T.RandomApply([T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))], p=0.5), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp24 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), rnd_color_jitter, T.RandomGrayscale(p=0.2), T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp25 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), rnd_color_jitter, T.RandomGrayscale(p=0.2), T.ToTensor(), T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.5)])
nat_transform_exp26 = T.Compose([T.RandomResizedCrop(32), T.RandomGrayscale(p=1.0), rnd_color_jitter, T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])

nat_transform_exp27 = T.Compose([T.RandomResizedCrop(32, scale=(0.8, 1.0)), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp28 = T.Compose([DiscreteRotation(), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])

# EXP 29 (N29) — THE ULTIMATE BEAST
# Chen et al. (2020): Full color-disruption + structural cutout.
# Rationale: ColorJitter forces the encoder to ignore color shortcuts.
# RandomErasing (Cutout) forces it to learn spatial structure.
# Together, these synergistically prevent ALL forms of shortcut learning.
nat_transform_exp29 = T.Compose([
    # 1. Base spatial: aggressive crop forces invariance to scale/position
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    # 2. Horizontal flip: invariance to left-right mirroring
    T.RandomHorizontalFlip(p=0.5),
    # 3. Color jitter (p=0.8): prevents color-histogram shortcut learning
    T.RandomApply([T.ColorJitter(0.8, 0.8, 0.8, 0.2)], p=0.8),
    # 4. Grayscale (p=0.2): forces encoder to occasionally rely on luminance only
    T.RandomGrayscale(p=0.2),
    # 5. Must convert to tensor BEFORE RandomErasing (requires tensor input)
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
    # 6. Cutout / Random Erasing: destroys local patches -> spatial robustness
    T.RandomErasing(p=0.5, scale=(0.05, 0.2)),
])

# ============================================================================
# NEW JITTERED EXPERIMENTS (Added 19 May 2026) - For 5 Top Models
# ============================================================================
# 1. Exp 38: Pure Rotation + Jitter (Base: Exp 36)
nat_transform_exp30 = T.Compose([
    T.RandomApply([T.ColorJitter(0.8, 0.8, 0.8, 0.2)], p=0.8),
    DiscreteRotation(), 
    T.ToTensor(), 
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# 2. Exp 39: Weak Baseline + Jitter (Base: Exp 35)
nat_transform_exp31 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.8, 1.0)), 
    T.RandomApply([T.ColorJitter(0.8, 0.8, 0.8, 0.2)], p=0.8),
    T.ToTensor(), 
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# 3. Exp 40: Crop + Blur + Jitter (Base: Exp 9)
nat_transform_exp32 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.6, 1.0)), 
    T.RandomApply([T.ColorJitter(0.8, 0.8, 0.8, 0.2)], p=0.8),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), 
    T.ToTensor(), 
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# 4. Exp 41: Crop + Flip + Blur + Jitter (Base: Exp 13)
nat_transform_exp33 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.6, 1.0)), 
    T.RandomHorizontalFlip(), 
    T.RandomApply([T.ColorJitter(0.8, 0.8, 0.8, 0.2)], p=0.8),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), 
    T.ToTensor(), 
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# 5. Exp 42: Crop + Cutout + Jitter (Base: Exp 10)
nat_transform_exp34 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.6, 1.0)), 
    T.RandomApply([T.ColorJitter(0.8, 0.8, 0.8, 0.2)], p=0.8),
    T.ToTensor(), 
    T.Normalize(CIFAR_MEAN, CIFAR_STD), 
    T.RandomErasing(p=0.7, scale=(0.02, 0.2), ratio=(0.3, 3.3))
])



## 3. Dataset Loaders & Subsets
Handles the standard SimCLR view generation and the SupCon stratified 10% subsets.


In [3]:

import os
import numpy as np
from torchvision.datasets import CIFAR10
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as T

# ══════════════════════════════════════════════════════════════════════════
# 1. REGISTRY 1 — Midterm 8-Experiment Ablation Configs
# ══════════════════════════════════════════════════════════════════════════

EXP_REGISTRY = {
    1: {
        "transform":   transform_exp1,
        "name":        "Exp1 — Crop only",
        "description": "Spatial only: RandomResizedCrop.",
    },
    2: {
        "transform":   transform_exp2,
        "name":        "Exp2 — Crop + Flip",
        "description": "Spatial: RandomResizedCrop + HorizontalFlip.",
    },
    3: {
        "transform":   transform_exp3,
        "name":        "Exp3 — Crop + ColorJitter",
        "description": "Hybrid: RandomResizedCrop + ColorJitter.",
    },
    4: {
        "transform":   transform_exp4,
        "name":        "Exp4 — Crop + Grayscale",
        "description": "Hybrid: RandomResizedCrop + RandomGrayscale.",
    },
    5: {
        "transform":   transform_exp5,
        "name":        "Exp5 — Crop + Flip + Color",
        "description": "Hybrid: RandomResizedCrop + Flip + ColorJitter.",
    },
    6: {
        "transform":   transform_exp6,
        "name":        "Exp6 — Crop + Flip + Grayscale",
        "description": "Hybrid: RandomResizedCrop + Flip + Grayscale.",
    },
    7: {
        "transform":   transform_exp7,
        "name":        "Exp7 — Crop + Color + Grayscale",
        "description": "Hybrid: RandomResizedCrop + ColorJitter + Grayscale.",
    },
    8: {
        "transform":   transform_exp8,
        "name":        "Exp8 — Full SimCLR",
        "description": "Full pipeline: Crop + Flip + ColorJitter + Grayscale.",
    },
    # NATALIE'S 28 EXPERIMENTS (Offset by 8)
    9:  {"transform": nat_transform_exp1,  "name": "Exp9(N1) — Crop + Blur", "description": "Single: Crop + Gaussian Blur."},
    10: {"transform": nat_transform_exp2,  "name": "Exp10(N2) — Crop + Cutout", "description": "Single: Crop + Random Erasing."},
    11: {"transform": nat_transform_exp3,  "name": "Exp11(N3) — Crop + Sobel", "description": "Single: Crop + Sobel Edge Extraction."},
    12: {"transform": nat_transform_exp4,  "name": "Exp12(N4) — Crop + Noise", "description": "Single: Crop + Gaussian Noise."},
    13: {"transform": nat_transform_exp5,  "name": "Exp13(N5) — Crop + Flip + Blur", "description": "Dual: Spatial + Blur."},
    14: {"transform": nat_transform_exp6,  "name": "Exp14(N6) — Crop + Flip + Cutout", "description": "Dual: Spatial + Corruption."},
    15: {"transform": nat_transform_exp7,  "name": "Exp15(N7) — Crop + Flip + Sobel", "description": "Dual: Spatial + Structural."},
    16: {"transform": nat_transform_exp8,  "name": "Exp16(N8) — Crop + Flip + Noise", "description": "Dual: Spatial + Noise."},
    17: {"transform": nat_transform_exp9,  "name": "Exp17(N9) — Crop + Color + Blur", "description": "Dual: Photometric + Blur."},
    18: {"transform": nat_transform_exp10, "name": "Exp18(N10) — Crop + Color + Cutout", "description": "Dual: Photometric + Corruption."},
    19: {"transform": nat_transform_exp11, "name": "Exp19(N11) — Crop + Color + Sobel", "description": "Dual: Photometric + Structural."},
    20: {"transform": nat_transform_exp12, "name": "Exp20(N12) — Crop + Color + Noise", "description": "Dual: Photometric + Noise."},
    21: {"transform": nat_transform_exp13, "name": "Exp21(N13) — Crop + Gray + Blur", "description": "Dual: Grayscale + Blur."},
    22: {"transform": nat_transform_exp14, "name": "Exp22(N14) — Crop + Gray + Cutout", "description": "Dual: Grayscale + Corruption."},
    23: {"transform": nat_transform_exp15, "name": "Exp23(N15) — Crop + Gray + Sobel", "description": "Dual: Grayscale + Structural."},
    24: {"transform": nat_transform_exp16, "name": "Exp24(N16) — Crop + Gray + Noise", "description": "Dual: Grayscale + Noise."},
    25: {"transform": nat_transform_exp17, "name": "Exp25(N17) — Crop + Blur + Sobel", "description": "Conflict: Smooth vs Sharp Edge."},
    26: {"transform": nat_transform_exp18, "name": "Exp26(N18) — Crop + Flip + Blur + Cutout", "description": "Conflict: Content Erasure with Blur."},
    27: {"transform": nat_transform_exp19, "name": "Exp27(N19) — Crop + Sobel + Noise", "description": "Conflict: Edge Extraction vs Pixel Noise."},
    28: {"transform": nat_transform_exp20, "name": "Exp28(N20) — Crop + Blur + Noise", "description": "Conflict: High-frequency Distortion (JPEG-like)."},
    29: {"transform": nat_transform_exp21, "name": "Exp29(N21) — Crop + Flip + Color + Erasing", "description": "Conflict: Color Distortion with Patch Erasure."},
    30: {"transform": nat_transform_exp22, "name": "Exp30(N22) — Crop + Heavy Cutout Grid", "description": "Extreme Spatial Corruption Baseline."},
    31: {"transform": nat_transform_exp23, "name": "Exp31(N23) — SimCLR Classic Strong", "description": "Standard SimCLR Pipeline (Chen et al. 2020)."},
    32: {"transform": nat_transform_exp24, "name": "Exp32(N24) — SimCLR + Rotation Task", "description": "Pipeline for E-SSL Joint-Task Pretraining."},
    33: {"transform": nat_transform_exp25, "name": "Exp33(N25) — Full Heatmap Suite", "description": "All Invariance Transforms Active Simultaneously."},
    34: {"transform": nat_transform_exp26, "name": "Exp34(N26) — Color Destruction", "description": "Strict Color Elimination: Grayscale p=1.0 + ColorJitter."},
    35: {"transform": nat_transform_exp27, "name": "Exp35(N27) — Weak Baseline", "description": "Minimal Spatial Invariance Only."},
    36: {"transform": nat_transform_exp28, "name": "Exp36(N28) — Pure Discrete Rotation", "description": "Isolated Structural Rotation Alignment."},
    # ── Exp 29 (N29) — The Ultimate Beast (added 18 May 2026) ──────────────────
    37: {
        "transform":   nat_transform_exp29,
        "name":        "Exp37(N29) — The Ultimate Beast",
        "description": (
            "Full pipeline: Crop + Flip + ColorJitter(p=0.8) + Grayscale(p=0.2) "
            "+ RandomErasing/Cutout(p=0.5, scale=5-20%). "
            "Designed to eliminate ALL shortcut-learning pathways simultaneously."
        ),
    },
    # ── Re-runs with Color Jitter (Added 19 May 2026) ──────────────────────────
    38: {"transform": nat_transform_exp30, "name": "Exp38 — Pure Rotation + Jitter (Base: Exp 36)", "description": "Exp 36 + Color Jitter"},
    39: {"transform": nat_transform_exp31, "name": "Exp39 — Weak Baseline + Jitter (Base: Exp 35)", "description": "Exp 35 + Color Jitter"},
    40: {"transform": nat_transform_exp32, "name": "Exp40 — Crop + Blur + Jitter (Base: Exp 9)", "description": "Exp 9 + Color Jitter"},
    41: {"transform": nat_transform_exp33, "name": "Exp41 — Crop + Flip + Blur + Jitter (Base: Exp 13)", "description": "Exp 13 + Color Jitter"},
    42: {"transform": nat_transform_exp34, "name": "Exp42 — Crop + Cutout + Jitter (Base: Exp 10)", "description": "Exp 10 + Color Jitter"},
}


def _base_crop():
    """All new augs build on top of the base SimCLR crop."""
    return T.RandomResizedCrop(32, scale=(0.2, 1.0))

# ── Placeholder entries — Natalie fills in the transform pipelines ─────────
# For each entry, replace T.Compose([_base_crop(), T.ToTensor(), ...])
# with your actual pipeline.

EXTENDED_REGISTRY = {

    # ── Natalie: fill these in ────────────────────────────────────────

    "randaugment": {
        "transform": T.Compose([
            _base_crop(),
            T.RandAugment(num_ops=2, magnitude=9),   # NATALIE: tune params
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "RandAugment",
        "description": "AutoML-style random augmentation policy search.",
    },

    "autoaugment": {
        "transform": T.Compose([
            _base_crop(),
            T.AutoAugment(policy=T.AutoAugmentPolicy.CIFAR10),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "AutoAugment (CIFAR-10 policy)",
        "description": "Learned CIFAR-10 augmentation policy (Cubuk et al.).",
    },

    "trivialaugment": {
        "transform": T.Compose([
            _base_crop(),
            T.TrivialAugmentWide(),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "TrivialAugmentWide",
        "description": "Parameter-free augmentation (Müller & Hutter, 2021).",
    },

    "solarize": {
        "transform": T.Compose([
            _base_crop(),
            T.RandomHorizontalFlip(),
            T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            T.RandomSolarize(threshold=128, p=0.2),   # NATALIE: tune threshold
            T.RandomGrayscale(p=0.2),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "Full SimCLR + Solarize",
        "description": "Exp8 + RandomSolarize (inspired by SimCLRv2).",
    },

    "equalize": {
        "transform": T.Compose([
            _base_crop(),
            T.RandomHorizontalFlip(),
            T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            T.RandomEqualize(p=0.2),
            T.RandomGrayscale(p=0.2),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "Full SimCLR + Equalize",
        "description": "Exp8 + RandomEqualize histogram normalization.",
    },

    "sharpness": {
        "transform": T.Compose([
            _base_crop(),
            T.RandomHorizontalFlip(),
            T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            T.RandomAdjustSharpness(sharpness_factor=2, p=0.3),
            T.RandomGrayscale(p=0.2),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "Full SimCLR + Sharpness",
        "description": "Exp8 + RandomAdjustSharpness.",
    },

    "rotation": {
        "transform": T.Compose([
            _base_crop(),
            T.RandomHorizontalFlip(),
            T.RandomRotation(degrees=15),
            T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            T.RandomGrayscale(p=0.2),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "Full SimCLR + Rotation",
        "description": "Exp8 + RandomRotation ±15°.",
    },

    "perspective": {
        "transform": T.Compose([
            _base_crop(),
            T.RandomHorizontalFlip(),
            T.RandomPerspective(distortion_scale=0.2, p=0.3),
            T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            T.RandomGrayscale(p=0.2),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "Full SimCLR + Perspective",
        "description": "Exp8 + RandomPerspective distortion.",
    },

    # NOTE: Mixup and CutMix require batch-level operations — they are
    # implemented in the training loop via torchvision.transforms.v2.
    # Natalie: implement these as collate_fn wrappers in augmentations.py
    # and register them here by referencing your collate_fn.
    # Mahmoud's DataLoader accepts an optional collate_fn parameter.
    "mixup": {
        "transform": transform_exp8,   # NATALIE: replace with Mixup-aware transform
        "name":        "Mixup (placeholder)",
        "description": "Batch-level Mixup — requires custom collate_fn.",
        "collate_fn":  None,           # NATALIE: set to your mixup_collate_fn
    },

    "cutmix": {
        "transform": transform_exp8,   # NATALIE: replace with CutMix-aware transform
        "name":        "CutMix (placeholder)",
        "description": "Batch-level CutMix — requires custom collate_fn.",
        "collate_fn":  None,           # NATALIE: set to your cutmix_collate_fn
    },
}


# ══════════════════════════════════════════════════════════════════════════
# 3. AugmentedDataset — returns (view1, view2), label
# ══════════════════════════════════════════════════════════════════════════

class AugmentedDataset(Dataset):
    """
    Wraps any torchvision dataset to return two augmented views.
    Compatible with both NT-Xent (labels ignored) and SupCon (labels used).
    """
    def __init__(self, dataset, transform):
        self.dataset   = dataset
        self.transform = SimCLRViewGenerator(transform, n_views=2)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        views = self.transform(img)      # [view_1, view_2]
        return (views[0], views[1]), label


# ══════════════════════════════════════════════════════════════════════════
# 4. DATALOADER FACTORIES
# ══════════════════════════════════════════════════════════════════════════

def _resolve_transform(exp_id: int, aug_name: str):
    """
    Resolves which transform to use.
    Priority: aug_name (extended registry) > exp_id (midterm registry)
    """
    if aug_name is not None:
        aug_name = aug_name.lower().strip()
        if aug_name not in EXTENDED_REGISTRY:
            available = list(EXTENDED_REGISTRY.keys())
            raise ValueError(
                f"Unknown aug_name='{aug_name}'. "
                f"Available: {available}"
            )
        info = EXTENDED_REGISTRY[aug_name]
        print(f"  [Dataset] Augmentation (extended): {info['name']}")
        print(f"            {info['description']}")
        return info["transform"], info.get("collate_fn", None)

    # Fallback to midterm exp_id
    if exp_id not in EXP_REGISTRY:
        raise ValueError(
            f"Invalid exp_id={exp_id}. Choose from {list(EXP_REGISTRY.keys())}."
        )
    info = EXP_REGISTRY[exp_id]
    print(f"  [Dataset] Augmentation (midterm): {info['name']}")
    print(f"            {info['description']}")
    return info["transform"], None


def get_train_dataloader(
    data_dir: str,
    batch_size: int   = 128,
    num_workers: int  = 4,
    shuffle: bool     = True,
    exp_id: int       = 8,
    aug_name: str     = None,
) -> DataLoader:
    """
    Returns a training DataLoader yielding ((view1, view2), label).

    Args:
        data_dir    : root directory for CIFAR-10 download
        batch_size  : training batch size
        num_workers : DataLoader worker processes
        shuffle     : shuffle training data
        exp_id      : midterm ablation config 1–8 (overridden by aug_name)
        aug_name    : named augmentation from EXTENDED_REGISTRY
    """
    transform, collate_fn = _resolve_transform(exp_id, aug_name)

    os.makedirs(data_dir, exist_ok=True)
    base_dataset = CIFAR10(root=data_dir, train=True, download=True)
    augmented    = AugmentedDataset(base_dataset, transform)

    loader_kwargs = dict(
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True,      # ensures consistent batch size for NT-Xent
        persistent_workers=(num_workers > 0),
    )
    if collate_fn is not None:
        loader_kwargs["collate_fn"] = collate_fn

    return DataLoader(augmented, **loader_kwargs)


def get_supcon_dataloader(
    data_dir: str,
    batch_size: int     = 256,
    num_workers: int    = 4,
    label_fraction: float = 0.1,
    exp_id: int         = 8,
    aug_name: str       = None,
    seed: int           = 42,
) -> DataLoader:
    """
    Returns a labeled DataLoader for Bonus #4 (SupCon training).
    Subsamples `label_fraction` of the CIFAR-10 training set, stratified
    per class so class balance is maintained.

    Args:
        label_fraction: 0.1 = 10% = 5,000 images (500 per class)
    """
    transform, collate_fn = _resolve_transform(exp_id, aug_name)
    rng = np.random.RandomState(seed)

    os.makedirs(data_dir, exist_ok=True)
    base_dataset  = CIFAR10(root=data_dir, train=True, download=True)
    labels_array  = np.array(base_dataset.targets)

    # Stratified subsampling
    selected = []
    for cls in range(10):
        cls_idx = np.where(labels_array == cls)[0].tolist()
        n_keep  = max(1, int(len(cls_idx) * label_fraction))
        chosen  = rng.choice(cls_idx, size=n_keep, replace=False).tolist()
        selected.extend(chosen)

    print(f"  [SupCon] {len(selected):,} labeled images "
          f"({label_fraction*100:.0f}% of 50k, stratified per class)")

    subset    = Subset(base_dataset, selected)
    augmented = AugmentedDataset(subset, transform)

    loader_kwargs = dict(
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True,
        persistent_workers=(num_workers > 0),
    )
    if collate_fn is not None:
        loader_kwargs["collate_fn"] = collate_fn

    return DataLoader(augmented, **loader_kwargs)


def get_eval_dataloader(
    data_dir: str,
    train: bool      = False,
    batch_size: int  = 256,
    num_workers: int = 4,
) -> DataLoader:
    """
    Returns an evaluation DataLoader (no augmentation, no shuffling).
    Used for: t-SNE generation, linear probe caching, FAISS index building.
    """
    os.makedirs(data_dir, exist_ok=True)

    eval_transform = T.Compose([
        T.ToTensor(),
        T.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    dataset = CIFAR10(
        root=data_dir, train=train, download=True, transform=eval_transform
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=(num_workers > 0),
    )



In [4]:
import numpy as np
import torch
from torch.utils.data import Subset
from torchvision.datasets import CIFAR10


CIFAR10_CLASSES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

NUM_CLASSES = 10
SAMPLES_PER_CLASS = 500  # 10% of 5000 per class


def get_stratified_subset(
    dataset,
    fraction=0.1,
    num_classes=10,
    random_seed=42,
    verbose=True
):
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)
    
    # ── Extract targets from dataset ─────────────────────────────────
    if hasattr(dataset, 'targets'):
        targets = np.array(dataset.targets)
    elif hasattr(dataset, 'labels'):
        targets = np.array(dataset.labels)
    else:
        raise ValueError(
            f"Dataset {type(dataset)} has no 'targets' or 'labels' attribute. "
            "Cannot extract labels for stratification."
        )
    
    total_samples = len(dataset)
    samples_per_class = int(total_samples * fraction / num_classes)
    
    indices = []
    
    # ── Stratify: sample exactly `samples_per_class` from each class ──
    for class_idx in range(num_classes):
        # Find all indices with this class label
        class_indices = np.where(targets == class_idx)[0]
        
        # Randomly sample `samples_per_class` from this class
        sampled_indices = np.random.choice(
            class_indices,
            size=min(samples_per_class, len(class_indices)),
            replace=False
        )
        
        indices.extend(sampled_indices)
    
    indices = np.array(indices)
    
    # ── Shuffle the combined indices ─────────────────────────────────
    np.random.shuffle(indices)
    
    # ── Create Subset ────────────────────────────────────────────────
    subset = Subset(dataset, indices.tolist())
    
    # ── Print statistics ─────────────────────────────────────────────
    if verbose:
        print("\n" + "="*70)
        print("  STRATIFIED SUBSET SAMPLING")
        print("="*70)
        print(f"  Original dataset size:  {total_samples:,} samples")
        print(f"  Subset fraction:        {fraction:.1%}")
        print(f"  Subset size:            {len(subset):,} samples")
        print(f"  Samples per class:      {samples_per_class:,}")
        print(f"  Number of classes:      {num_classes}")
        print()
        
        # Verify balance
        subset_targets = targets[indices]
        for class_idx in range(num_classes):
            count = np.sum(subset_targets == class_idx)
            class_name = CIFAR10_CLASSES[class_idx] if num_classes == 10 else f"class_{class_idx}"
            print(f"    {class_name:15s}: {count:4d} samples")
        
        print("="*70 + "\n")
    
    return subset


def get_balanced_dataloader(
    dataset,
    fraction=0.1,
    batch_size=512,
    num_workers=4,
    shuffle=True,
    random_seed=42
):
   
    from torch.utils.data import DataLoader
    
    subset = get_stratified_subset(
        dataset,
        fraction=fraction,
        random_seed=random_seed
    )
    
    dataloader = DataLoader(
        subset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True  # Ensures all batches have exactly batch_size samples
    )
    
    return dataloader, subset


# ─────────────────────────────────────────────────────────────────
# TEST & VERIFICATION
# ─────────────────────────────────────────────────────────────────

if False:  # [Notebook] __main__ disabled
    """
    Quick test to verify stratified sampling works correctly.
    
    Run with: python dataset_subset.py
    """
    
    from torchvision.datasets import CIFAR10
    from torch.utils.data import DataLoader
    
    # Download CIFAR-10 (if not already)
    print("Loading CIFAR-10...")
    train_set = CIFAR10('./data', train=True, download=True)
    
    # Create 10% stratified subset
    print("\nCreating 10% stratified subset...")
    subset = get_stratified_subset(train_set, fraction=0.1, verbose=True)
    
    # Verify with DataLoader
    print("Creating DataLoader from subset...")
    dataloader = DataLoader(
        subset,
        batch_size=512,
        shuffle=True,
        num_workers=0,  # Use 0 for testing
        drop_last=True
    )
    
    print(f"Number of batches: {len(dataloader)}")
    
    # Check first batch
    images, labels = next(iter(dataloader))
    print(f"\nFirst batch:")
    print(f"  Images shape: {images.shape}")
    print(f"  Labels shape: {labels.shape}")
    print(f"  Unique labels in batch: {torch.unique(labels).tolist()}")
    print(f"  Label counts in batch:")
    for class_idx in range(10):
        count = (labels == class_idx).sum().item()
        if count > 0:
            print(f"    Class {class_idx}: {count}")
    
    print("\n Stratified sampling test passed!")



## 4. Architectures
ResNet-18 and ResNet-50 with CIFAR-10 stem modifications and the non-linear Projection Head.


In [5]:
import torch
import torch.nn as nn
import torchvision.models as models


class ProjectionHead(nn.Module):

    def __init__(
        self,
        input_dim=512,
        hidden_dim=512,
        output_dim=128
    ):

        super().__init__()

        self.layers = nn.Sequential(

            nn.Linear(
                input_dim,
                hidden_dim,
                bias=False
            ),

            nn.BatchNorm1d(hidden_dim),

            nn.ReLU(inplace=True),

            nn.Linear(
                hidden_dim,
                output_dim
            )
        )

    def forward(self, x):

        return self.layers(x)


class SimCLRResNet18(nn.Module):

    def __init__(self, projection_dim=128):

        super().__init__()

        backbone = models.resnet18(weights=None)

        # adjust first layer for CIFAR-10 images
        backbone.conv1 = nn.Conv2d(
            3,
            64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        # remove original maxpool
        backbone.maxpool = nn.Identity()

        self.encoder = nn.Sequential(

            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,

            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,

            backbone.avgpool
        )

        self.feature_dim = 512

        self.projector = ProjectionHead(
            input_dim=self.feature_dim,
            hidden_dim=self.feature_dim,
            output_dim=projection_dim
        )

    def forward(self, x):

        features = self.encoder(x)

        features = torch.flatten(
            features,
            start_dim=1
        )

        projections = self.projector(features)

        return features, projections


class SimCLRResNet50(nn.Module):
    """
    ResNet-50 backbone for SimCLR with CIFAR-10 stem adjustment.
    Output: (h, z) where h is the 2048-dim hidden representation
    and z is the 128-dim projection (or custom projection_dim).
    """

    def __init__(self, projection_dim=128):

        super().__init__()

        backbone = models.resnet50(weights=None)

        # adjust first layer for CIFAR-10 images (32x32)
        backbone.conv1 = nn.Conv2d(
            3,
            64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        # remove original maxpool
        backbone.maxpool = nn.Identity()

        self.encoder = nn.Sequential(

            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,

            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,
        )

        # global average pooling
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        # projection head: ResNet50 outputs 2048-dim features
        self.projection = ProjectionHead(
            input_dim=2048,
            hidden_dim=2048,
            output_dim=projection_dim
        )

    def forward(self, x):
        """
        Args:
            x: (N, 3, 32, 32) CIFAR-10 batch
        Returns:
            h: (N, 2048) encoder output
            z: (N, projection_dim) projection output
        """
        x = self.encoder(x)            # (N, 2048, 1, 1)
        x = self.pool(x)               # (N, 2048, 1, 1)
        h = x.flatten(start_dim=1)     # (N, 2048)
        z = self.projection(h)         # (N, projection_dim)
        return h, z


def get_simclr_model(
    backbone="resnet18",
    projection_dim=128
):

    if backbone == "resnet18":

        return SimCLRResNet18(
            projection_dim=projection_dim
        )

    elif backbone == "resnet50":

        return SimCLRResNet50(
            projection_dim=projection_dim
        )

    raise ValueError(
        f"Backbone '{backbone}' is not supported. "
        f"Supported: resnet18, resnet50"
    )

## 5. Loss Functions
NT-Xent (SimCLR) and SupConLoss (Supervised Contrastive).

> **Note:** SupConLoss is defined twice (custom + HobbitLong official). The second definition (official, temperature=0.1) overwrites the first and is the one used by `train_supcon()`.


In [6]:

import torch
import torch.nn as nn
import torch.nn.functional as F


# ══════════════════════════════════════════════════════════════════════
# 1. NT-Xent Loss  (unchanged from midterm, kept here for single import)
# ══════════════════════════════════════════════════════════════════════

class NTXentLoss(nn.Module):
    """
    Normalized Temperature-scaled Cross Entropy loss.
    For a batch of N images → 2N views (two augmented copies each).
    Each view's positive is its twin; all 2(N-1) other views are negatives.

    Args:
        temperature (float): τ scaling factor. Default 0.5 (paper default).
    """

    def __init__(self, temperature: float = 0.5):
        super().__init__()
        self.temperature = temperature

    def forward(self, z1: torch.Tensor, z2: torch.Tensor) -> torch.Tensor:
        """
        Args:
            z1, z2: projection embeddings, shape (N, D). NOT yet normalized.
        Returns:
            Scalar NT-Xent loss averaged over both views.
        """
        N = z1.shape[0]
        device = z1.device

        # ℓ₂-normalize so cosine sim = dot product
        z1 = F.normalize(z1, dim=1)
        z2 = F.normalize(z2, dim=1)

        # Stack into (2N, D) — [z1_0, z1_1, …, z2_0, z2_1, …]
        z = torch.cat([z1, z2], dim=0)                         # (2N, D)

        # Full similarity matrix scaled by temperature
        sim = torch.mm(z, z.T) / self.temperature             # (2N, 2N)

        # Mask out self-similarity (diagonal)
        diag_mask = torch.eye(2 * N, dtype=torch.bool, device=device)
        sim = sim.masked_fill(diag_mask, float("-inf"))

        # Positive targets: view i pairs with view i+N (and vice versa)
        targets = torch.cat([
            torch.arange(N, 2 * N, device=device),
            torch.arange(0, N, device=device),
        ])

        return F.cross_entropy(sim, targets)


# ══════════════════════════════════════════════════════════════════════
# 2. SupCon Loss  (Bonus #4)  — Khosla et al., NeurIPS 2020
# ══════════════════════════════════════════════════════════════════════

class SupConLoss(nn.Module):

    def __init__(
        self,
        temperature: float = 0.07,
        contrast_mode: str = "all",
        base_temperature: float = 0.07,
    ):
        super().__init__()
        assert temperature > 0, "Temperature must be positive."
        assert contrast_mode in ("all", "one"), \
            "contrast_mode must be 'all' or 'one'."

        self.temperature = temperature
        self.contrast_mode = contrast_mode
        self.base_temperature = base_temperature

    def forward(
        self,
        features: torch.Tensor,
        labels: torch.Tensor = None,
        mask: torch.Tensor = None,
    ) -> torch.Tensor:

        device = features.device

        # ── Input validation ─────────────────────────────────────────
        if features.ndim == 2:
            # Caller passed (N, D) — assume single view, unsqueeze
            features = features.unsqueeze(1)

        batch_size, n_views, feat_dim = features.shape

        # ── Build positive-pair mask ──────────────────────────────────
        if mask is not None:
            # Explicit mask provided — trust the caller
            mask = mask.float().to(device)

        elif labels is not None:
            # Supervised: mask[i,j] = 1 iff label[i] == label[j]
            labels = labels.contiguous().view(-1, 1)          # (N, 1)
            mask = torch.eq(labels, labels.T).float().to(device)  # (N, N)

        else:
            # Self-supervised fallback: each sample is its own class
            mask = torch.eye(batch_size, dtype=torch.float32, device=device)

        # ── Flatten views into one big (N*n_views, D) tensor ─────────
        # Anchor features — determined by contrast_mode
        if self.contrast_mode == "one":
            anchor_feat = features[:, 0, :]                   # (N, D)
            anchor_count = 1
        else:  # "all"
            # Stack all views: each becomes an anchor
            anchor_feat = features.view(
                batch_size * n_views, feat_dim
            )                                                  # (N*v, D)
            anchor_count = n_views

        # Contrast features = ALL views of ALL samples
        contrast_feat = features.view(
            batch_size * n_views, feat_dim
        )                                                      # (N*v, D)
        contrast_count = n_views

        # ── Similarity matrix ─────────────────────────────────────────
        # anchor_dot_contrast: (N*anchor_count, N*contrast_count)
        anchor_dot_contrast = (
            torch.mm(anchor_feat, contrast_feat.T) / self.temperature
        )

        # For numerical stability: subtract row-wise max
        logits_max, _ = anchor_dot_contrast.max(dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        # ── Expand mask to cover all view combinations ─────────────────
        # Original mask is (N, N). We tile it to (N*anchor_count, N*contrast_count)
        # so that every view of sample i treats every view of a same-class
        # sample j as a positive.
        mask_tiled = mask.repeat(anchor_count, contrast_count)   # (N*a, N*c)

        # Remove self-contrast (an anchor must not compare with itself)
        self_contrast_mask = torch.scatter(
            torch.ones_like(mask_tiled),
            1,
            torch.arange(batch_size * anchor_count, device=device).view(-1, 1),
            0,
        )
        mask_tiled = mask_tiled * self_contrast_mask

        # ── Log-softmax over negatives ────────────────────────────────
        exp_logits = torch.exp(logits) * self_contrast_mask    # zero out self
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-9)

        # ── Mean over positives ───────────────────────────────────────
        # For each anchor, average log-prob over its positives
        mean_log_prob_pos = (mask_tiled * log_prob).sum(1) / (
            mask_tiled.sum(1) + 1e-9
        )

        # ── Final loss (temperature re-scaling as in paper Eq. 2) ─────
        loss = -(self.temperature / self.base_temperature) * mean_log_prob_pos
        loss = loss.view(anchor_count, batch_size).mean()

        return loss


# ══════════════════════════════════════════════════════════════════════
# 3. Loss Factory  — used by train_master.py and train_supcon.py
# ══════════════════════════════════════════════════════════════════════

def get_loss(loss_type: str, temperature: float = 0.5) -> nn.Module:
    """
    Factory function so all training scripts import losses by name.

    Args:
        loss_type   : 'ntxent' | 'supcon'
        temperature : τ value (same for both losses)
    Returns:
        Instantiated loss module.
    """
    if loss_type == "ntxent":
        return NTXentLoss(temperature=temperature)

    if loss_type == "supcon":
        # SupCon paper uses τ=0.07 by default; we allow override
        return SupConLoss(
            temperature=temperature,
            contrast_mode="all",
            base_temperature=0.07,
        )

    raise ValueError(
        f"Unknown loss '{loss_type}'. Choose 'ntxent' or 'supcon'."
    )


# ══════════════════════════════════════════════════════════════════════
# 4. Smoke Tests
# ══════════════════════════════════════════════════════════════════════

if False:  # [Notebook] __main__ disabled
    import math

    print("=" * 60)
    print("Smoke Test — NTXentLoss")
    print("=" * 60)

    crit = NTXentLoss(temperature=0.5)
    N, D = 32, 128

    z1 = torch.randn(N, D)
    z2 = torch.randn(N, D)
    loss_rand = crit(z1, z2)
    loss_expected = math.log(2 * N - 1)

    print(f"Random embeddings loss : {loss_rand.item():.4f}")
    print(f"Expected (uniform)     : {loss_expected:.4f}")

    loss_perfect = crit(z1, z1.clone())
    print(f"Identical embeddings   : {loss_perfect.item():.6f}  (should be ~0)")
    assert loss_rand.shape == torch.Size([]), "Not a scalar!"
    print("PASSED ✓\n")

    # ──────────────────────────────────────────────────────────────────
    print("=" * 60)
    print("Smoke Test — SupConLoss (supervised mode, Bonus #4)")
    print("=" * 60)

    supcon = SupConLoss(temperature=0.07)
    N_sup = 16
    n_views = 2
    D_sup = 128
    n_classes = 10

    # Simulate two augmented views: features shape (N, n_views, D)
    feats = F.normalize(
        torch.randn(N_sup, n_views, D_sup), dim=2
    )
    labels = torch.randint(0, n_classes, (N_sup,))

    loss_sup = supcon(feats, labels=labels)
    print(f"SupCon loss (supervised, 2 views) : {loss_sup.item():.4f}")
    assert loss_sup.shape == torch.Size([]), "Not a scalar!"
    assert not torch.isnan(loss_sup), "NaN detected!"
    print("PASSED ✓\n")

    # Self-supervised fallback (no labels)
    loss_self = supcon(feats, labels=None)
    print(f"SupCon loss (self-supervised mode) : {loss_self.item():.4f}")
    assert not torch.isnan(loss_self), "NaN detected!"
    print("PASSED ✓\n")

    print("=" * 60)
    print("All loss tests passed.")
    print("=" * 60)

print('✅ Cell 5a/9 complete: NTXentLoss + SupConLoss (custom) defined')


✅ Cell 5a/9 complete: NTXentLoss + SupConLoss (custom) defined


In [7]:
"""
loss_supcon.py — Supervised Contrastive Loss (Khosla et al., 2020)
───────────────────────────────────────────────────────────────────
Official implementation from: https://github.com/HobbitLong/SupContrast

Mathematical formulation (Khosla et al., NeurIPS 2020, Eq. 2):

    L_sup = (1/N) Σᵢ  (-1/|P(i)|) Σₚ∈P(i)  log [
        exp(zᵢ·zₚ/τ) / Σₐ∈A(i) exp(zᵢ·zₐ/τ)
    ]

Where:
  • zᵢ, zₚ: normalized embeddings
  • P(i): positive set (same class as i, excluding i)
  • A(i): all indices except i (in-batch negatives)
  • τ: temperature scaling (default 0.1, per paper Section 4.5)
"""

import torch
import torch.nn as nn


class SupConLoss(nn.Module):

    def __init__(self, temperature=0.1, contrast_mode='all',
                 base_temperature=0.1):
        super(SupConLoss, self).__init__()
        self.temperature = temperature
        self.contrast_mode = contrast_mode
        self.base_temperature = base_temperature

    def forward(self, features, labels=None, mask=None):
        """
        Args:
            features: hidden vector of shape [bsz, n_views, ...].
            labels: ground truth of shape [bsz].
            mask: contrastive mask of shape [bsz, bsz], mask_{i,j}=1 if sample j
                has the same class as sample i. Can be asymmetric.
        Returns:
            A loss scalar.
        """
        device = features.device

        if len(features.shape) < 3:
            raise ValueError('`features` needs to be [bsz, n_views, ...],'
                             ' at least 3 dimensions are required')
        if len(features.shape) > 3:
            features = features.view(features.shape[0], features.shape[1], -1)

        batch_size = features.shape[0]
        if labels is not None and mask is not None:
            raise ValueError('Cannot define both `labels` and `mask`')
        elif labels is None and mask is None:
            mask = torch.eye(batch_size, dtype=torch.float32).to(device)
        elif labels is not None:
            labels = labels.contiguous().view(-1, 1)
            if labels.shape[0] != batch_size:
                raise ValueError('Num of labels does not match num of features')
            mask = torch.eq(labels, labels.T).float().to(device)
        else:
            mask = mask.float().to(device)

        contrast_count = features.shape[1]
        contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)
        if self.contrast_mode == 'one':
            anchor_feature = features[:, 0]
            anchor_count = 1
        elif self.contrast_mode == 'all':
            anchor_feature = contrast_feature
            anchor_count = contrast_count
        else:
            raise ValueError('Unknown mode: {}'.format(self.contrast_mode))

        # compute logits
        anchor_dot_contrast = torch.div(
            torch.matmul(anchor_feature, contrast_feature.T),
            self.temperature)
        # for numerical stability
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        # tile mask
        mask = mask.repeat(anchor_count, contrast_count)
        # mask-out self-contrast cases
        logits_mask = torch.scatter(
            torch.ones_like(mask),
            1,
            torch.arange(batch_size * anchor_count).view(-1, 1).to(device),
            0
        )
        mask = mask * logits_mask

        # compute log_prob
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True))

        # compute mean of log-likelihood over positive
        # Modified to handle edge case where mask.sum(1) could be 0
        mask_pos_pairs = mask.sum(1)
        mask_pos_pairs = torch.where(
            mask_pos_pairs < 1e-6, torch.ones_like(mask_pos_pairs), mask_pos_pairs
        )
        mean_log_prob_pos = (mask * log_prob).sum(1) / mask_pos_pairs

        # loss
        loss = -(self.temperature / self.base_temperature) * mean_log_prob_pos
        loss = loss.view(anchor_count, batch_size).mean()

        return loss



## 6. Training Engines
Here we define the training loops for SimCLR and SupCon.


In [8]:


import os
import sys
import time

# Windows console encoding fix (tau character support)
pass  # [Notebook] reconfigure disabled
pass  # [Notebook] reconfigure disabled
import math
import json
# [Notebook] argparse removed
# [Notebook] subprocess removed
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib
# [Notebook] Agg backend removed (Jupyter uses inline)
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
# [Notebook] Using torch.amp API directly
from sklearn.manifold import TSNE


# ══════════════════════════════════════════════════════════════════════════
# 1. ARGUMENT PARSER
# ══════════════════════════════════════════════════════════════════════════

def parse_args():
    p = argparse.ArgumentParser(
        description="SimCLR Final-Phase Training — Group 20"
    )

    # ── Data ──────────────────────────────────────────────────────────
    p.add_argument("--data_dir",      type=str,   default="./data")
    p.add_argument("--output_dir",    type=str,   default="./outputs")
    p.add_argument("--exp_id",        type=int,   default=8,
        help="Augmentation config 1–8 (midterm ablations). "
             "Ignored if --aug_name is set.")
    p.add_argument("--aug_name",      type=str,   default=None,
        help="Named augmentation from Natalie's extended registry "
             "(e.g. 'randaugment', 'cutmix'). Overrides --exp_id.")

    # ── Model ─────────────────────────────────────────────────────────
    p.add_argument("--backbone",      type=str,   default="resnet50",
        choices=["resnet18", "resnet50"])
    p.add_argument("--projection_dim", type=int,  default=128)

    # ── Loss ──────────────────────────────────────────────────────────
    p.add_argument("--loss",          type=str,   default="ntxent",
        choices=["ntxent", "supcon"],
        help="'ntxent' for standard SimCLR. "
             "'supcon' for Bonus #4 hybrid training.")
    p.add_argument("--temperature",   type=float, default=0.5,
        help="NT-Xent τ. Sweep {0.1, 0.5, 1.0}.")
    p.add_argument("--label_fraction", type=float, default=0.1,
        help="Fraction of labels used for SupCon. Only active when "
             "--loss supcon. Default 0.10 = 10%%.")

    # ── Training ──────────────────────────────────────────────────────
    p.add_argument("--epochs",        type=int,   default=200)
    p.add_argument("--batch_size",    type=int,   default=1024)
    p.add_argument("--lr",            type=float, default=0.03,
        help="Peak LR for cosine schedule. "
             "Linear scaling rule: lr = 0.03 * batch_size / 256")
    p.add_argument("--auto_lr",       action="store_true", default=True,
        help="Apply linear LR scaling rule automatically.")
    p.add_argument("--weight_decay",  type=float, default=1e-4)
    p.add_argument("--warmup_epochs", type=int,   default=10,
        help="Linear warmup. Paper uses 10 epochs for 200-epoch runs.")
    p.add_argument("--optimizer",     type=str,   default="adamw",
        choices=["adamw", "sgd", "lars"],
        help="'lars' recommended for very large batches (BS >= 4096). "
             "'adamw' works well for BS 1024 on CIFAR-10.")

    # ── Checkpointing ─────────────────────────────────────────────────
    p.add_argument("--save_every",    type=int,   default=50,
        help="Save checkpoint every N epochs. "
             "Always saves at 50/100/150/200 regardless.")
    p.add_argument("--resume",        type=str,   default=None,
        help="Path to checkpoint .pth to resume from.")

    # ── Compute ───────────────────────────────────────────────────────
    p.add_argument("--num_workers",   type=int,   default=4)
    p.add_argument("--seed",          type=int,   default=42)
    p.add_argument("--no_compile",    action="store_true", default=False,
        help="Disable torch.compile (useful for debugging).")

    # ── Convenience ───────────────────────────────────────────────────
    p.add_argument("--run_all_ablations", action="store_true", default=False,
        help="Run all 8 midterm experiments sequentially in subprocesses.")
    p.add_argument("--tsne_final_only",   action="store_true", default=True)

    return p.parse_args()


# ══════════════════════════════════════════════════════════════════════════
# 2. REPRODUCIBILITY
# ══════════════════════════════════════════════════════════════════════════

def set_seed(seed: int):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        cudnn.deterministic = True


# ══════════════════════════════════════════════════════════════════════════
# 3. LEARNING RATE SCHEDULE
# ══════════════════════════════════════════════════════════════════════════

def build_scheduler(optimizer, warmup_epochs, total_epochs, steps_per_epoch):
    """
    Linear warmup followed by cosine annealing (step-level granularity).
    Using step-level (not epoch-level) ensures smooth LR even for large batches
    where each epoch has very few steps.
    """
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps  = total_epochs  * steps_per_epoch

    def lr_lambda(step):
        if step < warmup_steps:
            # Linear ramp from 0 → 1
            return float(step) / max(warmup_steps, 1)
        # Cosine annealing from 1 → 0
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ══════════════════════════════════════════════════════════════════════════
# 4. OPTIMIZER FACTORY
# ══════════════════════════════════════════════════════════════════════════

def build_optimizer(model, args):
    """
    Three optimizer options:
      adamw — best for BS 512–1024, stable, our primary choice
      sgd   — used in the original SimCLR paper with LARS for large batches
      lars  — wraps SGD with layer-adaptive rate scaling; needed for BS >=4096
    """
    params = model.parameters()

    if args.optimizer == "adamw":
        return optim.AdamW(params, lr=args.lr, weight_decay=args.weight_decay)

    if args.optimizer == "sgd":
        return optim.SGD(
            params, lr=args.lr,
            momentum=0.9, weight_decay=args.weight_decay, nesterov=True
        )

    if args.optimizer == "lars":
        # LARS is not in core PyTorch — use a pure-Python wrapper
        # that applies layer-wise adaptive rate scaling on top of SGD.
        try:
            from torch.optim import SGD

            class LARS(optim.Optimizer):
                """Simplified LARS (Layerwise Adaptive Rate Scaling)."""
                def __init__(self, params, lr, momentum=0.9,
                             weight_decay=1e-6, eta=1e-3):
                    defaults = dict(lr=lr, momentum=momentum,
                                    weight_decay=weight_decay, eta=eta)
                    super().__init__(params, defaults)

                @torch.no_grad()
                def step(self, closure=None):
                    loss = None
                    if closure is not None:
                        with torch.enable_grad():
                            loss = closure()
                    for group in self.param_groups:
                        for p in group["params"]:
                            if p.grad is None:
                                continue
                            param_norm = p.data.norm(2)
                            grad_norm  = p.grad.norm(2)
                            if param_norm > 0 and grad_norm > 0:
                                adaptive_lr = (
                                    group["eta"] * param_norm / (
                                        grad_norm
                                        + group["weight_decay"] * param_norm
                                        + 1e-9
                                    )
                                )
                                adaptive_lr = min(adaptive_lr, group["lr"])
                            else:
                                adaptive_lr = group["lr"]
                            d_p = p.grad + group["weight_decay"] * p.data
                            if "momentum_buffer" not in self.state[p]:
                                self.state[p]["momentum_buffer"] = torch.zeros_like(p.data)
                            buf = self.state[p]["momentum_buffer"]
                            buf.mul_(group["momentum"]).add_(d_p)
                            p.data.add_(buf, alpha=-adaptive_lr)
                    return loss

            return LARS(params, lr=args.lr, weight_decay=args.weight_decay)

        except Exception as e:
            print(f"[WARN] LARS build failed ({e}), falling back to AdamW.")
            return optim.AdamW(params, lr=args.lr,
                               weight_decay=args.weight_decay)

    raise ValueError(f"Unknown optimizer '{args.optimizer}'.")


# ══════════════════════════════════════════════════════════════════════════
# 5. CHECKPOINT UTILITIES
# ══════════════════════════════════════════════════════════════════════════

def save_checkpoint(state: dict, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(state, path)
    print(f"  [Checkpoint] Saved → {path}")


def load_checkpoint(path: str, model, optimizer, scheduler, scaler, device):
    print(f"  [Resume] Loading checkpoint from {path}")
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if "scaler_state_dict" in ckpt:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_loss   = ckpt.get("best_loss", float("inf"))
    loss_history = ckpt.get("loss_history", [])
    print(f"  [Resume] Resuming from epoch {start_epoch}, best loss {best_loss:.4f}")
    return start_epoch, best_loss, loss_history


# ══════════════════════════════════════════════════════════════════════════
# 6. t-SNE VISUALIZATION
# ══════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def generate_tsne(model, device, data_dir, epoch, output_dir,
                  num_workers=4, run_id="run"):
    print(f"\n  [t-SNE] Generating for epoch {epoch}...")
    model.eval()

    loader = get_eval_dataloader(
        data_dir, train=False, batch_size=512, num_workers=num_workers
    )
    all_feats, all_labels = [], []
    for imgs, lbls in loader:
        feats, _ = model(imgs.to(device))
        all_feats.append(feats.cpu().numpy())
        all_labels.append(lbls.numpy())
        if sum(x.shape[0] for x in all_feats) >= 2000:
            break

    feats  = np.concatenate(all_feats)[:2000]
    labels = np.concatenate(all_labels)[:2000]
    
    if np.isnan(feats).any():
        print("[t-SNE] Skipped because of NaN features")
        return

    emb = TSNE(
        n_components=2,
        random_state=42,
        perplexity=30
    ).fit_transform(feats)

    class_names = [
        "airplane", "automobile", "bird", "cat", "deer",
        "dog", "frog", "horse", "ship", "truck"
    ]
    fig, ax = plt.subplots(figsize=(10, 8))
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=labels, cmap="tab10",
                    alpha=0.6, s=8)
    cb = plt.colorbar(sc, ax=ax, ticks=range(10))
    cb.ax.set_yticklabels(class_names)
    ax.set_title(f"t-SNE — Epoch {epoch}  ({run_id})", fontsize=13,
                 fontweight="bold")
    ax.set_xticks([]); ax.set_yticks([])

    save_path = os.path.join(output_dir, "plots", f"tsne_epoch_{epoch:03d}.png")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  [t-SNE] Saved → {save_path}")
    model.train()


# ══════════════════════════════════════════════════════════════════════════
# 7. LOSS CURVE PLOT
# ══════════════════════════════════════════════════════════════════════════

def plot_loss_curve(loss_history, best_loss, output_dir, run_id):
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(range(1, len(loss_history) + 1), loss_history,
            lw=2, marker="o", markersize=4)
    ax.axhline(y=best_loss, ls="--", alpha=0.6,
               label=f"Best loss = {best_loss:.4f}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("NT-Xent / SupCon Loss")
    ax.set_title(f"Training Loss — {run_id}", fontsize=13, fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)
    path = os.path.join(output_dir, "plots", "training_loss_curve.png")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  [Plot] Loss curve saved → {path}")


# ══════════════════════════════════════════════════════════════════════════
# 8. DATASET HELPERS FOR SUPCON (10% LABEL FILTERING)
# ══════════════════════════════════════════════════════════════════════════

def get_supcon_dataloader(data_dir, batch_size, num_workers,
                          label_fraction=0.1, exp_id=8, seed=42):
    """
    For Bonus #4: returns a DataLoader that yields
        (view1, view2), label
    but only for `label_fraction` of CIFAR-10 training images
    (stratified per class so class balance is maintained).
    """
    from torchvision.datasets import CIFAR10
    from torch.utils.data import DataLoader, Subset
    # [Notebook] Inlined -- import removed
    import random as _random

    rng = np.random.RandomState(seed)

    base = CIFAR10(root=data_dir, train=True, download=True)
    labels_array = np.array(base.targets)

    # Stratified sampling — keep `label_fraction` per class
    selected_indices = []
    for cls in range(10):
        cls_idx = np.where(labels_array == cls)[0].tolist()
        n_keep  = max(1, int(len(cls_idx) * label_fraction))
        chosen  = rng.choice(cls_idx, size=n_keep, replace=False).tolist()
        selected_indices.extend(chosen)

    subset    = Subset(base, selected_indices)
    transform = EXP_REGISTRY[exp_id]["transform"]
    augmented = AugmentedDataset(subset, transform)

    print(f"  [SupCon] Using {len(selected_indices):,} images "
          f"({label_fraction*100:.0f}% of CIFAR-10)")

    return DataLoader(
        augmented,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True,
    )


# ══════════════════════════════════════════════════════════════════════════
# 9. RUN ALL ABLATIONS (convenience wrapper)
# ══════════════════════════════════════════════════════════════════════════

def run_all_ablations(args):
    """
    Spawns 8 subprocesses — one per midterm augmentation config.
    Each uses --exp_id 1..8 and the epoch/batch settings you passed.
    Useful for regenerating all 8 t-SNE plots and loss curves at once.
    """
    print("\n" + "=" * 60)
    print("Running all 8 ablation experiments sequentially")
    print("=" * 60)

    for exp in range(1, 9):
        print(f"\n{'─'*50}")
        print(f"  Ablation Exp {exp}/8")
        print(f"{'─'*50}")

        cmd = [
            sys.executable, r'd:\\Deep Learning Project\\Working in the Project Mahmoud\\Final\\SimCLR-Vision-SSL\\src\\train_master.py',
            "--exp_id",       str(exp),
            "--epochs",       str(args.epochs),
            "--batch_size",   str(args.batch_size),
            "--backbone",     args.backbone,
            "--loss",         "ntxent",
            "--temperature",  str(args.temperature),
            "--output_dir",   os.path.join(args.output_dir, f"exp_{exp}"),
            "--data_dir",     args.data_dir,
            "--num_workers",  str(args.num_workers),
            "--seed",         str(args.seed),
        ]
        if args.no_compile:
            cmd.append("--no_compile")

        try:
            subprocess.run(cmd, check=True)
            print(f"  [OK] Experiment {exp} finished.")
        except subprocess.CalledProcessError as e:
            print(f"  [ERROR] Experiment {exp} crashed — code {e.returncode}")

    print("\n" + "=" * 60)
    print("All 8 ablations done.")
    print("=" * 60)


# ══════════════════════════════════════════════════════════════════════════
# 10. MAIN TRAINING FUNCTION
# ══════════════════════════════════════════════════════════════════════════

def train(args):

    # ── Seed & cuDNN ────────────────────────────────────────────────
    set_seed(args.seed)
    cudnn.benchmark = True    # fastest conv algo selection

    # ── Run ID for organized output dirs ────────────────────────────
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    aug_tag = args.aug_name if args.aug_name else f"exp{args.exp_id}"
    run_id  = (
        f"{args.backbone}_{aug_tag}_"
        f"{args.loss}_τ{args.temperature}_"
        f"bs{args.batch_size}_ep{args.epochs}_{ts}"
    )
    output_dir = os.path.join(args.output_dir, run_id)

    for sub in ("checkpoints", "plots", "logs"):
        os.makedirs(os.path.join(output_dir, sub), exist_ok=True)

    # ── Save run config to JSON (for reproducibility) ───────────────
    config_path = os.path.join(output_dir, "run_config.json")
    with open(config_path, "w") as f:
        json.dump(vars(args), f, indent=2)
    print(f"\n  [Config] Saved run config → {config_path}")

    # ── Device ──────────────────────────────────────────────────────
    device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = device.type == "cuda"
    print(f"  [Device] {device}  |  AMP={use_amp}")
    if device.type == "cuda":
        print(f"  [GPU]    {torch.cuda.get_device_name(0)}")
        print(f"  [VRAM]   {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

    # ── Auto LR scaling ─────────────────────────────────────────────
    # Linear scaling rule: lr = base_lr × batch_size / 256
    # Reference: Goyal et al. (2017) "Accurate, Large Minibatch SGD"
    if args.auto_lr:
        args.lr = 0.03 * args.batch_size / 256
        print(f"  [LR]     Auto-scaled to {args.lr:.4f} "
              f"(0.03 × {args.batch_size} / 256)")

    # ── Data ────────────────────────────────────────────────────────
    if args.loss == "supcon":
        train_loader = get_supcon_dataloader(
            args.data_dir,
            args.batch_size,
            args.num_workers,
            label_fraction=args.label_fraction,
            exp_id=args.exp_id,
            seed=args.seed,
        )
    else:
        train_loader = get_train_dataloader(
            args.data_dir,
            batch_size=args.batch_size,
            num_workers=args.num_workers,
            exp_id=args.exp_id,
            aug_name=args.aug_name,   # None → falls back to exp_id
        )

    steps_per_epoch = len(train_loader)
    total_samples   = len(train_loader.dataset)

    print(f"\n  [Data]   {total_samples:,} training samples")
    print(f"           {steps_per_epoch} steps/epoch  |  BS={args.batch_size}")

    # ── Model ───────────────────────────────────────────────────────
    model = get_simclr_model(
        backbone=args.backbone,
        projection_dim=args.projection_dim,
    ).to(device)

    # torch.compile: TorchInductor fuses kernels → ~25-35% speedup on Ada
    # Disabled in first-run debugging with --no_compile
    if not args.no_compile and hasattr(torch, "compile"):
        try:
            model = torch.compile(model)
            print("  [Compile] torch.compile() enabled (TorchInductor backend)")
        except Exception as e:
            print(f"  [Compile] torch.compile skipped: {e}")
    else:
        print("  [Compile] torch.compile disabled")

    total_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  [Model]  {args.backbone} — {total_params:.2f}M parameters")

    # ── Loss ────────────────────────────────────────────────────────
    criterion = get_loss(args.loss, args.temperature).to(device)
    print(f"  [Loss]   {args.loss.upper()}  τ={args.temperature}")

    # ── Optimizer + Scheduler + Scaler ──────────────────────────────
    optimizer = build_optimizer(model, args)
    scheduler = build_scheduler(
        optimizer, args.warmup_epochs, args.epochs, steps_per_epoch
    )
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    # ── Resume from checkpoint ──────────────────────────────────────
    start_epoch  = 1
    best_loss    = float("inf")
    loss_history = []

    if args.resume:
        start_epoch, best_loss, loss_history = load_checkpoint(
            args.resume, model, optimizer, scheduler, scaler, device
        )

    # ── Training loop ───────────────────────────────────────────────
    log_path = os.path.join(output_dir, "logs", "training_log.csv")
    with open(log_path, "w") as f:
        f.write("epoch,loss,lr,time_s\n")

    print(f"\n{'─'*55}")
    print(f"  {'Epoch':>6}  {'Loss':>10}  {'LR':>12}  {'Time':>8}")
    print(f"{'─'*55}")

    # Fixed checkpoint milestone epochs for Mirna's linear probe
    MILESTONE_EPOCHS = {50, 100, 150, 200}

    for epoch in range(start_epoch, args.epochs + 1):
        model.train()
        running_loss = 0.0
        t0 = time.time()

        for batch in train_loader:
            (x1, x2), labels_batch = batch

            x1 = x1.to(device, non_blocking=True)
            x2 = x2.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                _, z1 = model(x1)
                _, z2 = model(x2)

                if args.loss == "ntxent":
                    loss = criterion(z1, z2)
                elif args.loss == "supcon":
                    import torch.nn.functional as F
                    z1n = F.normalize(z1, dim=1)
                    z2n = F.normalize(z2, dim=1)
                    features = torch.stack([z1n, z2n], dim=1)  # (N, 2, D)
                    lbl = labels_batch.to(device, non_blocking=True)
                    loss = criterion(features, labels=lbl)
                else:
                    raise ValueError(f"Unknown loss: {args.loss}")

                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"[WARN] NaN/Inf loss at epoch {epoch}, skipping batch")
                    optimizer.zero_grad(set_to_none=True)
                    continue

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)    
            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() == scale_before:
                scheduler.step()

            running_loss += loss.item()

        avg_loss   = running_loss / steps_per_epoch
        current_lr = optimizer.param_groups[0]["lr"]
        elapsed    = time.time() - t0

        loss_history.append(avg_loss)

        is_best = avg_loss < best_loss
        if is_best:
            best_loss = avg_loss

        print(f"  {epoch:>6}  {avg_loss:>10.4f}  {current_lr:>12.6f}  {elapsed:>7.1f}s"
              f"{'  ★' if is_best else ''}")

        with open(log_path, "a") as f:
            f.write(f"{epoch},{avg_loss:.6f},{current_lr:.8f},{elapsed:.2f}\n")

        # ── Save checkpoints ──────────────────────────────────────────
        # Save at: milestone epochs, best loss, user-defined interval
        should_save = (
            epoch in MILESTONE_EPOCHS
            or epoch % args.save_every == 0
            or epoch == args.epochs
            or is_best
        )

        if should_save:
            tag    = "BEST" if is_best else f"ep{epoch:03d}"
            ckpt_path = os.path.join(
                output_dir, "checkpoints",
                f"simclr_{args.backbone}_{aug_tag}_{tag}.pth"
            )
            save_checkpoint(
                {
                    "epoch": epoch,
                    "model_state_dict":     model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "scaler_state_dict":    scaler.state_dict(),
                    "loss": avg_loss,
                    "best_loss": best_loss,
                    "loss_history": loss_history,
                    "args": vars(args),
                    "run_id": run_id,
                },
                ckpt_path,
            )

        # ── t-SNE (final epoch only by default) ──────────────────────
        if not args.tsne_final_only or epoch == args.epochs:
            generate_tsne(
                model, device, args.data_dir, epoch,
                output_dir, args.num_workers, run_id
            )

    # ── Final loss curve ────────────────────────────────────────────
    plot_loss_curve(loss_history, best_loss, output_dir, run_id)

    # ── Save encoder-only weights (for Mirna + ONNX export) ─────────
    encoder_path = os.path.join(output_dir, "simclr_encoder_final.pth")
    torch.save(model.state_dict(), encoder_path)
    print(f"\n  [Export] Encoder weights → {encoder_path}")

    # ── Summary ─────────────────────────────────────────────────────
    print("\n" + "=" * 55)
    print(f"  Run complete: {run_id}")
    print(f"  Best loss  : {best_loss:.4f}")
    print(f"  Epochs     : {args.epochs}")
    print(f"  Log CSV    : {log_path}")
    print(f"  Checkpoints: {output_dir}/checkpoints/")
    print("=" * 55 + "\n")


# ══════════════════════════════════════════════════════════════════════════
# 11. ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════

if False:  # [Notebook] __main__ disabled
    args = parse_args()

    if args.run_all_ablations:
        run_all_ablations(args)
    else:
        train(args)



In [9]:


import os
import sys
import time
import json
# [Notebook] argparse removed
from pathlib import Path
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
# [Notebook] Using torch.amp API directly
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from torchvision import transforms

# Ensure we can import from src/
pass  # [Notebook] sys.path removed
pass  # [Notebook] sys.path removed

# [Notebook] Local imports inlined from earlier cells


# -- CIFAR-10 configuration -------------------------------------------
NUM_CLASSES = 10
CIFAR10_CLASSES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]


class AugmentedDataset(torch.utils.data.Dataset):
    """
    Wrapper to return two augmented views + labels for SupCon training.
    
    Each __getitem__ returns:
        (view_1, view_2, label)
    
    where view_1 and view_2 are independent augmentations of the same image.
    """
    
    def __init__(self, cifar_dataset, augmentation_transform):
        self.dataset = cifar_dataset
        self.augmentation = augmentation_transform
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        
        # Two independent augmentations
        view_1 = self.augmentation(image)
        view_2 = self.augmentation(image)
        
        return view_1, view_2, label


def build_augmentation():
    """
    Build the augmentation pipeline for SupCon training.
    
    Uses SimCLR's augmentation strategy:
      • RandomResizedCrop (0.2-1.0 scale)
      • RandomHorizontalFlip
      • RandomApply ColorJitter
      • RandomApply GaussianBlur (if available)
      • RandomGrayscale
      • Normalize with CIFAR-10 statistics
    """
    
    s = 0.5  # Color distortion strength (from SimCLR paper)
    
    augmentation = transforms.Compose([
        transforms.RandomResizedCrop(32, scale=(0.2, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply([
            transforms.ColorJitter(
                brightness=0.8*s,
                contrast=0.8*s,
                saturation=0.8*s,
                hue=0.2*s
            )
        ], p=0.8),
        transforms.RandomApply([
            transforms.GaussianBlur(kernel_size=3)
        ], p=0.1),
        transforms.RandomGrayscale(p=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
    ])
    
    return augmentation


def create_datasets(data_dir, fraction=0.1, batch_size=512, num_workers=4):
    """
    Create stratified training subset + DataLoader for SupCon.
    
    Args:
        data_dir: Where to download/cache CIFAR-10
        fraction: Fraction of data (0.1 = 10%)
        batch_size: Batch size
        num_workers: Number of workers
    
    Returns:
        train_loader: DataLoader
        subset: The Subset object (for statistics)
    """
    
    # Download CIFAR-10 (no transforms here, augmentation happens in AugmentedDataset)
    train_set = CIFAR10(
        data_dir,
        train=True,
        download=True,
        transform=None
    )
    
    # Create stratified subset (5000 samples for 10%)
    subset = get_stratified_subset(
        train_set,
        fraction=fraction,
        verbose=True
    )
    
    # Wrap with augmentation
    augmentation = build_augmentation()
    augmented_subset = AugmentedDataset(subset, augmentation)
    
    # Create DataLoader
    train_loader = DataLoader(
        augmented_subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True
    )
    
    return train_loader, subset


def build_model_and_optimizer(learning_rate, use_lars=True):
    """
    Build ResNet-50 encoder + projection head + optimizer.
    
    Uses SGD + momentum 0.9 (Khosla et al., 2020, Section 4.1):
      • LR = 0.05 for batch_size=512
      • Weight decay = 1e-4
      • Momentum = 0.9
    
    Args:
        learning_rate: Base learning rate (0.05 for batch 512)
        use_lars: Use LARS optimizer (recommended for batch >= 4096)
    
    Returns:
        encoder: ResNet-50 backbone
        optimizer: SGD or LARS
    """
    
    # ResNet-50 encoder
    encoder = SimCLRResNet50(projection_dim=128)
    encoder.eval()  # Will be set to train() in training loop
    
    # Collect parameters
    params = list(encoder.parameters())
    
    if use_lars:
        try:
            from torchlars import LARS
            base_optimizer = optim.SGD(
                params,
                lr=learning_rate,
                momentum=0.9,
                weight_decay=1e-4
            )
            optimizer = LARS(base_optimizer)
        except ImportError:
            print("⚠️ torchlars not found. Using SGD + momentum (SupCon default).")
            optimizer = optim.SGD(
                params,
                lr=learning_rate,
                momentum=0.9,
                weight_decay=1e-4
            )
    else:
        # SGD + momentum 0.9 (matches SupCon paper exactly)
        optimizer = optim.SGD(
            params,
            lr=learning_rate,
            momentum=0.9,
            weight_decay=1e-4
        )
    
    return encoder, optimizer


def build_scheduler(optimizer, total_epochs, warmup_epochs=10):
    """Build cosine annealing scheduler (warmup handled per-step in train_epoch)."""
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=total_epochs, eta_min=0
    )
    return scheduler


def warmup_learning_rate(warmup_epochs, warmup_from, warmup_to,
                         epoch, batch_id, total_batches, optimizer):
    """Per-step linear warmup (matching official SupContrast repo)."""
    if epoch <= warmup_epochs:
        p = (batch_id + (epoch - 1) * total_batches) / \
            (warmup_epochs * total_batches)
        lr = warmup_from + p * (warmup_to - warmup_from)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr


def train_epoch(
    epoch,
    train_loader,
    encoder,
    criterion,
    optimizer,
    warmup_epochs,
    warmup_from,
    warmup_to,
    device
):
    """
    Single epoch of SupCon training (FP32).
    Per-step warmup during first warmup_epochs (matching official repo).
    """
    
    encoder.train()
    loss_meter = 0.0
    time_start = time.time()
    total_batches = len(train_loader)
    
    for batch_idx, (view1, view2, labels) in enumerate(train_loader):
        view1 = view1.to(device)
        view2 = view2.to(device)
        labels = labels.to(device)
        
        # Per-step warmup (official SupContrast approach)
        warmup_learning_rate(warmup_epochs, warmup_from, warmup_to,
                             epoch, batch_idx, total_batches, optimizer)
        
        optimizer.zero_grad()
        
        # Forward pass (FP32)
        _, z1 = encoder(view1)
        _, z2 = encoder(view2)
        
        # L2-normalize projections
        z1 = torch.nn.functional.normalize(z1, dim=1)
        z2 = torch.nn.functional.normalize(z2, dim=1)
        
        # Stack for SupConLoss: [N, 2, 128]
        features = torch.stack([z1, z2], dim=1)
        
        # Compute loss
        loss = criterion(features, labels)
        
        # Backward
        loss.backward()
        optimizer.step()
        
        loss_meter += loss.item()
    
    time_elapsed = time.time() - time_start
    loss_avg = loss_meter / len(train_loader)
    
    return loss_avg, time_elapsed


def train(args):
    """
    Main training loop for SupCon Stage 1.
    """
    
    # -- Setup --------------------------------------------------------
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    cudnn.benchmark = True
    
    # Create output directory
    run_id = f"supcon_resnet50_frac{int(args.fraction*100)}_bs{args.batch_size}_ep{args.epochs}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    output_dir = Path(f"./outputs/{run_id}")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    (output_dir / "checkpoints").mkdir(exist_ok=True)
    (output_dir / "logs").mkdir(exist_ok=True)
    (output_dir / "plots").mkdir(exist_ok=True)
    
    print(f"\n{'='*70}")
    print(f"  SUPERVISED CONTRASTIVE LEARNING (SupCon) -- STAGE 1")
    print(f"  Khosla et al. (2020)")
    print(f"{'='*70}")
    print(f"\n  [Config] Run ID: {run_id}")
    print(f"  [Device] {device}")
    if device.type == 'cuda':
        print(f"  [GPU]    {torch.cuda.get_device_name(0)}")
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  [VRAM]   {vram:.1f} GB")
    
    # -- Create datasets and loaders ----------------------------------
    print(f"\n  [Data] Loading CIFAR-10 ({args.fraction:.1%} subset)...")
    train_loader, subset = create_datasets(
        args.data_dir,
        fraction=args.fraction,
        batch_size=args.batch_size,
        num_workers=args.num_workers
    )
    print(f"         Subset size: {len(subset):,} samples")
    print(f"         Batch size: {args.batch_size}")
    print(f"         Steps/epoch: {len(train_loader)}")
    
    # -- Build model --------------------------------------------------
    print(f"\n  [Model] ResNet-50 + 128-dim projection")
    encoder, optimizer = build_model_and_optimizer(
        args.learning_rate,
        use_lars=args.use_lars
    )
    encoder = encoder.to(device)
    
    print(f"         Parameters: {sum(p.numel() for p in encoder.parameters()) / 1e6:.2f}M")
    
    # -- Loss and scheduler -------------------------------------------
    criterion = SupConLoss(temperature=0.1)  # Paper Section 4.5: "All our results used τ=0.1"
    
    warmup_epochs = 10
    warmup_from = 0.01
    scheduler = build_scheduler(optimizer, args.epochs, warmup_epochs=warmup_epochs)
    
    # Compute warmup target LR (from official SupContrast repo)
    import math
    warmup_to = args.learning_rate * (
        1 + math.cos(math.pi * warmup_epochs / args.epochs)) / 2
    
    print(f"  [Loss]  SupConLoss (tau={criterion.temperature})")
    print(f"  [Optim] {'LARS' if args.use_lars else 'SGD'} (momentum=0.9, wd=1e-4)")
    print(f"  [Sched] Cosine annealing + warmup ({warmup_epochs} epochs, from {warmup_from})")
    
    # -- Training loop ------------------------------------------------
    print(f"\n{'-'*70}")
    print(f"   Epoch      Loss          LR        Time")
    print(f"{'-'*70}")
    
    log_data = []
    best_loss = float('inf')
    best_epoch = 0
    
    for epoch in range(1, args.epochs + 1):
        loss, time_taken = train_epoch(
            epoch,
            train_loader,
            encoder,
            criterion,
            optimizer,
            warmup_epochs,
            warmup_from,
            warmup_to,
            device
        )
        
        # Step cosine scheduler after warmup ends
        if epoch > warmup_epochs:
            scheduler.step()
        
        current_lr = optimizer.param_groups[0]['lr']
        
        # Logging
        log_data.append({
            'epoch': epoch,
            'loss': loss,
            'learning_rate': current_lr,
            'time': time_taken
        })
        
        # Print
        marker = "*" if loss < best_loss else " "
        print(f"   {epoch:3d}    {loss:.4f}     {current_lr:.6f}   {time_taken:6.1f}s  {marker}")
        
        # Save best checkpoint
        if loss < best_loss:
            best_loss = loss
            best_epoch = epoch
            torch.save(
                encoder.state_dict(),
                output_dir / "checkpoints" / "supcon_best.pth"
            )
        
        # Save periodic checkpoints
        if epoch % 50 == 0:
            torch.save(
                encoder.state_dict(),
                output_dir / "checkpoints" / f"supcon_epoch_{epoch:03d}.pth"
            )
    
    print(f"{'-'*70}\n")
    
    # -- Save results -------------------------------------------------
    print(f"  [Checkpoint] Best model > {output_dir}/checkpoints/supcon_best.pth")
    
    # Save encoder for linear evaluation
    torch.save(
        encoder.state_dict(),
        output_dir / "supcon_encoder_final.pth"
    )
    print(f"  [Encoder] Saved for Phase 2 > {output_dir}/supcon_encoder_final.pth")
    
    # Save CSV log
    import csv
    with open(output_dir / "logs" / "training_log.csv", 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['epoch', 'loss', 'learning_rate', 'time'])
        writer.writeheader()
        writer.writerows(log_data)
    print(f"  [Log] CSV saved > {output_dir}/logs/training_log.csv")
    
    # Save config
    config = vars(args)
    config['run_id'] = run_id
    config['best_loss'] = best_loss
    config['best_epoch'] = best_epoch
    with open(output_dir / "run_config.json", 'w') as f:
        json.dump(config, f, indent=2)
    
    print(f"\n{'='*70}")
    print(f"  ✅ SupCon training complete!")
    print(f"  Best loss: {best_loss:.4f} (epoch {best_epoch})")
    print(f"  Total epochs: {args.epochs}")
    print(f"  Output directory: {output_dir}")
    print(f"{'='*70}\n")


def parse_args():
    """Parse command-line arguments."""
    
    parser = argparse.ArgumentParser(
        description="SupCon Stage 1 Training -- Khosla et al. (2020)"
    )
    
    # Data
    parser.add_argument('--data_dir', type=str, default='./data',
                        help='Path to CIFAR-10 data')
    parser.add_argument('--fraction', type=float, default=0.1,
                        help='Fraction of data to use (0.1 = 10%%)')
    parser.add_argument('--num_workers', type=int, default=0,
                        help='DataLoader workers (0 for Windows compatibility)')
    
    # Model
    parser.add_argument('--backbone', type=str, default='resnet50',
                        help='Backbone architecture')
    
    # Training
    parser.add_argument('--epochs', type=int, default=200,
                        help='Number of epochs')
    parser.add_argument('--batch_size', type=int, default=512,
                        help='Batch size')
    parser.add_argument('--learning_rate', type=float, default=0.05,
                        help='Base learning rate (SupCon paper: 0.05 for batch 512)')
    parser.add_argument('--use_lars', action='store_true', default=False,
                        help='Use LARS optimizer (if available)')
    
    return parser.parse_args()


if False:  # [Notebook] __main__ disabled
    args = parse_args()
    train(args)



## 7. Evaluation & Linear Probes
Scripts for extracting representations and evaluating them via linear probing.

> **Note:** The checkpoint paths below are hardcoded to the original training machine. Update `base_dir` and checkpoint paths to match your environment before running.


In [10]:
import os
import sys
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision.transforms as T
from torchvision.datasets import CIFAR10

pass  # [Notebook] sys.path removed
# [Notebook] Inlined -- import removed

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class SimCLREncoderOnly(nn.Module):
    def __init__(self, simclr_model, repr_dim=2048):
        super().__init__()
        self.simclr_model = simclr_model
        self.repr_dim = getattr(simclr_model, 'repr_dim', repr_dim)

    def forward(self, x):
        h, z = self.simclr_model(x)
        return h

def extract_features(encoder, loader, device):
    encoder.eval()
    all_feats, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            out = encoder(imgs.to(device))
            feats = out[0] if isinstance(out, tuple) else out
            all_feats.append(feats.cpu())
            all_labels.append(lbls)
    return torch.cat(all_feats), torch.cat(all_labels)

def main():
    base_dir = os.path.dirname(os.path.abspath(r'd:\\Deep Learning Project\\Working in the Project Mahmoud\\Final Mahmoud Project SimCLR 20 May With All Features\\evaluate_all_probes.py'))
    
    checkpoints = [
        (36, "v2_pure_rotation", r"v2_pure_rotation\outputs\resnet50_exp36_ntxent_τ0.5_bs512_ep200_20260517_153808\simclr_encoder_final.pth"),
        (35, "v3_weak_baseline", r"v3_weak_baseline\outputs\resnet50_exp35_ntxent_τ0.5_bs512_ep200_20260517_174004\simclr_encoder_final.pth"),
        (9, "v4_crop_blur", r"v4_crop_blur\outputs\resnet50_exp9_ntxent_τ0.5_bs512_ep200_20260517_194204\simclr_encoder_final.pth"),
        (13, "v5_crop_flip_blur", r"v5_crop_flip_blur\outputs\resnet50_exp13_ntxent_τ0.5_bs512_ep200_20260517_214535\simclr_encoder_final.pth"),
        (10, "v6_crop_cutout", r"v6_crop_cutout\outputs\resnet50_exp10_ntxent_τ0.5_bs512_ep200_20260517_234848\simclr_encoder_final.pth")
    ]

    CIFAR_MEAN = [0.4914, 0.4822, 0.4465]
    CIFAR_STD  = [0.2023, 0.1994, 0.2010]
    test_transform = T.Compose([
        T.ToTensor(),
        T.Normalize(CIFAR_MEAN, CIFAR_STD)
    ])

    data_dir = os.path.join(base_dir, 'data')
    os.makedirs(data_dir, exist_ok=True)
    
    plain_train = DataLoader(
        CIFAR10(data_dir, train=True, download=True, transform=test_transform),
        batch_size=512, shuffle=False, num_workers=0, pin_memory=False
    )
    plain_test = DataLoader(
        CIFAR10(data_dir, train=False, download=True, transform=test_transform),
        batch_size=512, shuffle=False, num_workers=0, pin_memory=False
    )

    results = []

    for exp_id, folder, ckpt_rel in checkpoints:
        ckpt_path = os.path.join(base_dir, ckpt_rel)
        if not os.path.exists(ckpt_path):
            print(f"File not found: {ckpt_path}")
            continue
            
        print(f"\n=======================================================")
        print(f"Evaluating Experiment {exp_id} ({folder})")
        print(f"=======================================================")
        
        # Load model
        model = get_simclr_model(backbone='resnet50', projection_dim=128)
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        
        if 'model_state_dict' in ckpt:
            model.load_state_dict(ckpt['model_state_dict'])
        else:
            model.load_state_dict(ckpt)
        
        model.eval()
        FEATURE_DIM = getattr(model, 'repr_dim', 2048)
        encoder = SimCLREncoderOnly(model, repr_dim=FEATURE_DIM).to(device)
        
        print('Extracting train features...')
        train_feats, train_labels = extract_features(encoder, plain_train, device)
        print('Extracting test features...')
        test_feats, test_labels = extract_features(encoder, plain_test, device)
        
        probe_train_ds = TensorDataset(train_feats, train_labels)
        probe_test_ds  = TensorDataset(test_feats,  test_labels)

        probe_train_loader = DataLoader(probe_train_ds, batch_size=512, shuffle=True)
        probe_test_loader  = DataLoader(probe_test_ds,  batch_size=512, shuffle=False)

        probe = nn.Linear(FEATURE_DIM, 10).to(device)
        probe_criterion = nn.CrossEntropyLoss()
        probe_optimizer = optim.Adam(probe.parameters(), lr=1e-3, weight_decay=1e-4)

        PROBE_EPOCHS = 50
        print(f"{'Epoch':>6}  {'Train Acc':>9}  {'Test Acc':>8}")
        print('-' * 30)

        for ep in range(1, PROBE_EPOCHS + 1):
            probe.train()
            correct, total = 0, 0
            for feats, lbls in probe_train_loader:
                feats, lbls = feats.to(device), lbls.to(device)
                probe_optimizer.zero_grad()
                out  = probe(feats)
                loss = probe_criterion(out, lbls)
                loss.backward()
                probe_optimizer.step()
                correct += out.argmax(1).eq(lbls).sum().item()
                total   += lbls.size(0)
            tr_acc = 100.0 * correct / total
            
            probe.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for feats, lbls in probe_test_loader:
                    feats, lbls = feats.to(device), lbls.to(device)
                    out = probe(feats)
                    correct += out.argmax(1).eq(lbls).sum().item()
                    total   += lbls.size(0)
            te_acc = 100.0 * correct / total
            
            if ep % 10 == 0 or ep == 1:
                print(f"{ep:>6}  {tr_acc:>8.2f}%  {te_acc:>7.2f}%")

        probe_final_acc = te_acc
        print('-' * 30)
        print(f'Linear Probe Final Test Acc: {probe_final_acc:.2f}%')
        results.append((exp_id, folder, probe_final_acc))

    print("\n" + "="*60)
    print(" FINAL RESULTS SUMMARY (LINEAR PROBE - RESNET50) ")
    print("="*60)
    for r in results:
        print(f"Exp {r[0]:<2} | {r[1]:<20} | Top-1 Test Acc: {r[2]:.2f}%")
        
if False:  # [Notebook] __main__ disabled
    main()



## 8. CLIP Zero-Shot
Foundation model upper bound.

**Requirements:** `pip install transformers tqdm`


In [ ]:
# Auto-install CLIP dependencies into the current kernel's environment
import subprocess, importlib
for _pkg in ['transformers', 'tqdm']:
    try:
        importlib.import_module(_pkg)
    except ImportError:
        print(f'Installing {_pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _pkg])
print('Dependencies OK')


import sys
import time

# ═══════════════════════════════════════════════════════════════════════════
# Configuration
# ═══════════════════════════════════════════════════════════════════════════
MODEL_ID   = "openai/clip-vit-base-patch32"   # ViT-B/32, 151M params
BATCH_SIZE = 256
NUM_WORKERS = 4
DATA_ROOT  = "./data"                          # CIFAR-10 will download here

# CIFAR-10 class names (official order, index 0–9)
CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

# Classic CLIP zero-shot prompt template
PROMPT_TEMPLATE = "a photo of a {}"


def build_text_prompts(class_names: list[str]) -> list[str]:
    """Build the 10 zero-shot text prompts from class names."""
    return [PROMPT_TEMPLATE.format(name) for name in class_names]


def pil_collate_fn(batch):
    """
    Custom collate function that keeps images as raw PIL objects.
    
    The default collate would try to stack images into a tensor, but
    CLIPProcessor expects raw PIL images so it can apply its own
    internal resizing (224×224) and normalization.
    
    Returns:
        images: list[PIL.Image]   – raw PIL images
        labels: torch.Tensor      – class indices
    """
    images, labels = zip(*batch)
    return list(images), torch.tensor(labels, dtype=torch.long)


def main():
    # ── Device setup ────────────────────────────────────────────────────
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    
    print("═" * 72)
    print("  CLIP Zero-Shot Evaluation on CIFAR-10")
    print("═" * 72)
    print(f"  Model     : {MODEL_ID}")
    print(f"  Device    : {device} ({gpu_name})")
    print(f"  Batch Size: {BATCH_SIZE}")
    print(f"  Workers   : {NUM_WORKERS}")
    print("─" * 72)

    # ── Load CLIP model & processor ─────────────────────────────────────
    print("\n[1/3] Loading CLIP model and processor...")
    t0 = time.time()
    
    processor = CLIPProcessor.from_pretrained(MODEL_ID)
    model     = CLIPModel.from_pretrained(MODEL_ID).to(device)
    model.eval()
    
    num_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"      Model loaded in {time.time() - t0:.1f}s  "
          f"({num_params:.1f}M parameters)")

    # ── Prepare text embeddings (computed once) ─────────────────────────
    text_prompts = build_text_prompts(CIFAR10_CLASSES)
    print(f"\n[2/3] Text prompts ({len(text_prompts)} classes):")
    for i, prompt in enumerate(text_prompts):
        print(f"       [{i}] \"{prompt}\"")

    # ── Load CIFAR-10 test set ──────────────────────────────────────────
    #    transform=None → images are returned as raw PIL.Image objects
    print(f"\n[3/3] Loading CIFAR-10 test set...")
    
    test_dataset = CIFAR10(
        root=DATA_ROOT,
        train=False,
        download=True,
        transform=None,       # Raw PIL — CLIPProcessor handles everything
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        collate_fn=pil_collate_fn,
    )
    
    print(f"      {len(test_dataset):,} test images, "
          f"{len(test_loader)} batches of {BATCH_SIZE}")

    # ═══════════════════════════════════════════════════════════════════
    # Zero-Shot Inference Loop
    # ═══════════════════════════════════════════════════════════════════
    print("\n" + "─" * 72)
    print("  Running Zero-Shot Inference...")
    print("─" * 72)
    
    correct = 0
    total   = 0
    per_class_correct = [0] * len(CIFAR10_CLASSES)
    per_class_total   = [0] * len(CIFAR10_CLASSES)
    
    t_start = time.time()
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="  Evaluating",
                                    unit="batch", ncols=80):
            # ── Process batch: images (PIL) + text prompts → tensors ───
            inputs = processor(
                text=text_prompts,
                images=images,
                return_tensors="pt",
                padding=True,
            )
            # Move all tensors to GPU
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            # ── Forward pass ───────────────────────────────────────────
            outputs = model(**inputs)
            
            # logits_per_image: [batch_size, num_classes] — similarity
            # scores between each image and each text prompt
            logits = outputs.logits_per_image       # (B, 10)
            probs  = logits.softmax(dim=-1)          # (B, 10)
            preds  = probs.argmax(dim=-1).cpu()      # (B,)
            
            # ── Accumulate metrics ─────────────────────────────────────
            labels_cpu = labels
            correct += (preds == labels_cpu).sum().item()
            total   += labels_cpu.size(0)
            
            for pred, gt in zip(preds, labels_cpu):
                per_class_total[gt.item()] += 1
                if pred.item() == gt.item():
                    per_class_correct[gt.item()] += 1
    
    elapsed = time.time() - t_start
    accuracy = 100.0 * correct / total
    
    # ═══════════════════════════════════════════════════════════════════
    # Results Summary
    # ═══════════════════════════════════════════════════════════════════
    print("\n")
    print("╔" + "═" * 70 + "╗")
    print("║" + "  CLIP Zero-Shot Results (CIFAR-10 Test Set)".center(70) + "║")
    print("╠" + "═" * 70 + "╣")
    print("║" + f"  Model        : {MODEL_ID}".ljust(70) + "║")
    print("║" + f"  Device       : {gpu_name}".ljust(70) + "║")
    print("║" + f"  Test Images  : {total:,}".ljust(70) + "║")
    print("║" + f"  Inference    : {elapsed:.1f}s  "
          f"({total/elapsed:.0f} img/s)".ljust(70) + "║")
    print("╠" + "═" * 70 + "╣")
    print("║" + f"  ★  Top-1 Zero-Shot Accuracy:  {accuracy:.2f}%  "
          f"({correct}/{total})  ★".center(70) + "║")
    print("╠" + "═" * 70 + "╣")
    print("║" + "  Per-Class Breakdown:".ljust(70) + "║")
    print("║" + "  " + "─" * 50 + " " * 18 + "║")
    
    for i, cls_name in enumerate(CIFAR10_CLASSES):
        cls_acc = 100.0 * per_class_correct[i] / max(per_class_total[i], 1)
        bar_len = int(cls_acc / 2.5)   # scale to ~40 chars max
        bar = "█" * bar_len
        line = f"  {cls_name:<12s}  {cls_acc:5.1f}%  ({per_class_correct[i]:4d}/{per_class_total[i]:4d})  {bar}"
        print("║" + line.ljust(70) + "║")
    
    print("╚" + "═" * 70 + "╝")
    
    # ── Quick comparison context ────────────────────────────────────────
    print("\n  Context (CIFAR-10 Top-1 Accuracy):")
    print("  ┌────────────────────────────────────────────────────┐")
    print(f"  │  Our SimCLR (ResNet-50, 200ep) :  84.30%           │")
    print(f"  │  CLIP ViT-B/32 (Zero-Shot)    :  {accuracy:.2f}%           │")
    print(f"  │  Supervised CE (ResNet-50)     :  93.77%           │")
    print("  └────────────────────────────────────────────────────┘")
    
    return accuracy


if False:  # [Notebook] __main__ disabled
    accuracy = main()

print('✅ Cell 8/9 complete: CLIP zero-shot evaluation defined')


d:\Deep Learning Project\Working in the Project Mahmoud\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dependencies OK
✅ Cell 8/9 complete: CLIP zero-shot evaluation defined


## 9. Execution
Run your desired experiment by calling the corresponding functions below.

Uncomment the section you want to run.


In [ ]:
# ══════════════════════════════════════════════════════════════════
# OPTION A: Train SimCLR (Exp 41 Champion) 
# ══════════════════════════════════════════════════════════════════
# Uncomment and adjust epochs/batch_size as needed:
#
sys.argv = ['notebook']  # Reset argv for argparse
args = parse_args()
args.exp_id = 41
args.epochs = 200
args.batch_size = 512
args.backbone = 'resnet50'
args.loss = 'ntxent'
args.temperature = 0.5
args.lr = 3e-4
args.auto_lr = True
args.warmup_epochs = 10
args.weight_decay = 1e-4
args.optimizer = 'adamw'
args.projection_dim = 128
args.save_every = 50
args.resume = None
args.num_workers = 4
args.seed = 42
args.no_compile = True
args.run_all_ablations = False
args.tsne_final_only = True
args.aug_name = None
args.label_fraction = 0.1
args.data_dir = './data'
args.output_dir = './outputs'
train(args)







  SUPERVISED CONTRASTIVE LEARNING (SupCon) -- STAGE 1
  Khosla et al. (2020)

  [Config] Run ID: supcon_resnet50_frac10_bs512_ep200_20260609_102137
  [Device] cuda
  [GPU]    NVIDIA RTX 5000 Ada Generation
  [VRAM]   34.4 GB

  [Data] Loading CIFAR-10 (10.0% subset)...
Files already downloaded and verified


d:\Deep Learning Project\Working in the Project Mahmoud\venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")



  STRATIFIED SUBSET SAMPLING
  Original dataset size:  50,000 samples
  Subset fraction:        10.0%
  Subset size:            5,000 samples
  Samples per class:      500
  Number of classes:      10

    airplane       :  500 samples
    automobile     :  500 samples
    bird           :  500 samples
    cat            :  500 samples
    deer           :  500 samples
    dog            :  500 samples
    frog           :  500 samples
    horse          :  500 samples
    ship           :  500 samples
    truck          :  500 samples

         Subset size: 5,000 samples
         Batch size: 512
         Steps/epoch: 9

  [Model] ResNet-50 + 128-dim projection
         Parameters: 27.96M
  [Loss]  SupConLoss (tau=0.1)
  [Optim] SGD (momentum=0.9, wd=1e-4)
  [Sched] Cosine annealing + warmup (10 epochs, from 0.01)

----------------------------------------------------------------------
   Epoch      Loss          LR        Time
----------------------------------------------------------

In [ ]:
# ══════════════════════════════════════════════════════════════════
# OPTION B: Train SupCon on 10% labels
# ══════════════════════════════════════════════════════════════════
sys.argv = ['notebook']
supcon_args = parse_args()  # from train_supcon
supcon_args.epochs = 200
supcon_args.batch_size = 512
supcon_args.fraction = 0.1
supcon_args.data_dir = './data'
supcon_args.learning_rate = 0.05
supcon_args.use_lars = False
supcon_args.num_workers = 4
train(supcon_args)  # SupCon train function

In [ ]:
# ══════════════════════════════════════════════════════════════════
# OPTION C: Evaluate Linear Probes (requires trained checkpoints)
# ══════════════════════════════════════════════════════════════════
main()  # From evaluate_all_probes

In [ ]:

# ══════════════════════════════════════════════════════════════════
# OPTION D: CLIP Zero-Shot Evaluation
# ══════════════════════════════════════════════════════════════════
main()  # From evaluate_clip_zeroshot

print('✅ Cell 9/9 complete: execution cell ready')
